# Image Encoder Training — Multi-Modal Disease Detection

**Self-contained Kaggle notebook** that trains:
1. **EfficientNet-B4** image encoder (25-class multi-label) on NIH ChestX-ray14 + HAM10000 + Brain Tumor MRI
2. **MobileNetV3-Small** modality classifier (3-class: chest/skin/brain)

**Kaggle Datasets Required:**
- `nih-chest-xrays/data` (ChestX-ray14)
- `kmader/skin-cancer-mnist-ham10000`
- `sartajbhuvaji/brain-tumor-classification-mri`

**Outputs:**
- `image_encoder.pth`
- `modality_classifier.pth`
- `image_config.json`

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ── All hyperparameters in one place ──────────────────────────────────────────

CONFIG = {
    # Model
    'backbone': 'efficientnet_b4',
    'image_size': 380,
    'feature_dim': 1792,
    'projection_dim': 512,
    'num_classes': 25,
    'dropout': 0.3,

    # Training
    'batch_size': 16,
    'num_workers': 4,
    'epochs_frozen': 5,
    'epochs_unfrozen': 15,
    'lr_frozen': 1e-4,
    'lr_unfrozen': 1e-5,
    'weight_decay': 0.01,
    'focal_alpha': 0.75,
    'focal_gamma': 2.0,

    # Data
    'chest_sample_fraction': 0.15,
    'val_split': 0.15,
    'seed': SEED,

    # Labels
    'all_labels': [
        # Chest (0-13)
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
        'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
        'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
        'Pleural_Thickening', 'Hernia',
        # Skin (14-20)
        'Melanoma', 'Melanocytic_Nevi', 'Basal_Cell_Carcinoma',
        'Actinic_Keratosis', 'Benign_Keratosis', 'Dermatofibroma',
        'Vascular_Lesion',
        # Brain (21-24)
        'Glioma', 'Meningioma', 'Pituitary_Tumor', 'No_Tumor',
    ],
    'chest_labels': [
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
        'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
        'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
        'Pleural_Thickening', 'Hernia',
    ],
    'skin_labels': [
        'Melanoma', 'Melanocytic_Nevi', 'Basal_Cell_Carcinoma',
        'Actinic_Keratosis', 'Benign_Keratosis', 'Dermatofibroma',
        'Vascular_Lesion',
    ],
    'brain_labels': ['Glioma', 'Meningioma', 'Pituitary_Tumor', 'No_Tumor'],
    'skin_dx_to_label': {
        'mel': 'Melanoma', 'nv': 'Melanocytic_Nevi', 'bcc': 'Basal_Cell_Carcinoma',
        'akiec': 'Actinic_Keratosis', 'bkl': 'Benign_Keratosis',
        'df': 'Dermatofibroma', 'vasc': 'Vascular_Lesion',
    },
    'brain_folder_to_label': {
        'glioma_tumor': 'Glioma', 'meningioma_tumor': 'Meningioma',
        'pituitary_tumor': 'Pituitary_Tumor', 'no_tumor': 'No_Tumor',
    },

    # Paths (Kaggle)
    'chest_csv': '/kaggle/input/data/Data_Entry_2017_v2020.csv',
    'chest_image_dirs': [
        f'/kaggle/input/data/images_{str(i).zfill(3)}/images'
        for i in range(1, 13)
    ],
    'skin_csv': '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv',
    'skin_image_dirs': [
        '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
        '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2',
    ],
    'brain_root': '/kaggle/input/brain-tumor-classification-mri',
    'output_dir': '/kaggle/working',
}

print(f"Total classes: {CONFIG['num_classes']}")
print(f"Labels: {CONFIG['all_labels']}")

## 2. Model Definitions

In [ ]:
# ── Image Encoder ────────────────────────────────────────────────────────────

class ImageEncoder(nn.Module):
    def __init__(
        self,
        backbone_name: str = 'efficientnet_b4',
        pretrained: bool = True,
        feature_dim: int = 1792,
        projection_dim: int = 512,
        dropout: float = 0.3,
        freeze_backbone: bool = False,
    ):
        super().__init__()
        self.feature_dim = feature_dim
        self.projection_dim = projection_dim

        weights = 'IMAGENET1K_V1' if pretrained else None
        backbone = getattr(tv_models, backbone_name)(weights=weights)
        self.features = backbone.features
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        if freeze_backbone:
            for param in self.features.parameters():
                param.requires_grad = False

        self.projection = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def _extract_features(self, x):
        return self.features(x)

    def forward(self, x):
        feat = self._extract_features(x)
        pooled = self.global_pool(feat).flatten(1)
        return self.projection(pooled)

    def get_spatial_features(self, x):
        feat = self._extract_features(x)
        B, C, H, W = feat.shape
        feat = feat.reshape(B, C, H * W).permute(0, 2, 1)
        return self.projection(feat)

    def get_feature_map(self, x):
        return self._extract_features(x)

    def freeze_backbone(self):
        for param in self.features.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self, last_n_blocks: int = 2):
        blocks = list(self.features.children())
        for block in blocks[-last_n_blocks:]:
            for param in block.parameters():
                param.requires_grad = True


# ── Modality Classifier ────────────────────────────────────────────────────

class ModalityClassifier(nn.Module):
    def __init__(self, num_classes: int = 3, pretrained: bool = True, dropout: float = 0.2):
        super().__init__()
        weights = 'IMAGENET1K_V1' if pretrained else None
        backbone = tv_models.mobilenet_v3_small(weights=weights)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.classifier = nn.Sequential(
            nn.Linear(576, 256),
            nn.Hardswish(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat).flatten(1)
        return self.classifier(feat)


# ── Focal Loss ─────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = focal_weight * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# ── Image-only classification head ─────────────────────────────────────────

class ImageClassifier(nn.Module):
    """ImageEncoder + classification head for standalone training."""
    def __init__(self, encoder: ImageEncoder, num_classes: int = 25, dropout: float = 0.3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(encoder.projection_dim, encoder.projection_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(encoder.projection_dim // 2, num_classes),
        )

    def forward(self, x):
        emb = self.encoder(x)
        return self.head(emb)


print('Models defined.')

## 3. Data Processing

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_transforms(image_size=380, modality=None):
    augs = [T.Resize((image_size, image_size))]
    if modality != 'chest':
        augs.append(T.RandomVerticalFlip(p=0.3))
    augs.append(T.RandomHorizontalFlip(p=0.5))
    augs.append(T.RandomRotation(degrees=15))
    augs.append(T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)))
    if modality == 'skin':
        augs.append(T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1))
    elif modality == 'brain':
        augs.append(T.ColorJitter(brightness=0.15, contrast=0.15))
    else:
        augs.append(T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1))
    augs.extend([
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        T.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    ])
    return T.Compose(augs)

def get_val_transforms(image_size=380):
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

print('Transforms defined.')

In [ ]:
# ── Load and process ChestX-ray14 ─────────────────────────────────────────

def load_chest_data(config):
    """Load NIH ChestX-ray14, take a stratified sample."""
    csv_path = config['chest_csv']
    if not os.path.exists(csv_path):
        print(f'Chest CSV not found at {csv_path}, skipping chest data.')
        return []

    df = pd.read_csv(csv_path)
    print(f'Chest X-ray14: {len(df)} total images')

    # Build image path lookup
    image_lookup = {}
    for img_dir in config['chest_image_dirs']:
        if os.path.isdir(img_dir):
            for fname in os.listdir(img_dir):
                image_lookup[fname] = os.path.join(img_dir, fname)

    print(f'Found {len(image_lookup)} chest images on disk')

    # Stratified subsample
    sample_frac = config['chest_sample_fraction']
    df_sampled = df.sample(frac=sample_frac, random_state=SEED)
    print(f'Using {len(df_sampled)} chest images ({sample_frac*100:.0f}% sample)')

    records = []
    chest_labels = config['chest_labels']

    for _, row in df_sampled.iterrows():
        fname = row['Image Index']
        if fname not in image_lookup:
            continue

        # Multi-label: split by '|'
        findings = str(row['Finding Labels']).split('|')
        findings = [f.strip() for f in findings]

        # Build 25-class multi-hot vector
        label_vec = np.zeros(config['num_classes'], dtype=np.float32)
        for finding in findings:
            if finding == 'No Finding':
                continue
            finding_clean = finding.replace(' ', '_')
            if finding_clean in chest_labels:
                idx = chest_labels.index(finding_clean)
                label_vec[idx] = 1.0

        records.append({
            'image_path': image_lookup[fname],
            'labels': label_vec,
            'modality': 'chest',
        })

    print(f'Loaded {len(records)} valid chest records')
    return records


def load_skin_data(config):
    """Load HAM10000 skin lesion dataset."""
    csv_path = config['skin_csv']
    if not os.path.exists(csv_path):
        print(f'Skin CSV not found at {csv_path}, skipping skin data.')
        return []

    df = pd.read_csv(csv_path)
    print(f'HAM10000: {len(df)} total images')

    # Build image path lookup
    image_lookup = {}
    for img_dir in config['skin_image_dirs']:
        if os.path.isdir(img_dir):
            for fname in os.listdir(img_dir):
                name_no_ext = os.path.splitext(fname)[0]
                image_lookup[name_no_ext] = os.path.join(img_dir, fname)

    dx_map = config['skin_dx_to_label']
    skin_labels = config['skin_labels']
    skin_offset = 14  # skin labels start at index 14

    records = []
    for _, row in df.iterrows():
        image_id = row['image_id']
        if image_id not in image_lookup:
            continue

        dx = row['dx']
        if dx not in dx_map:
            continue

        label_name = dx_map[dx]
        label_vec = np.zeros(config['num_classes'], dtype=np.float32)
        label_vec[skin_offset + skin_labels.index(label_name)] = 1.0

        records.append({
            'image_path': image_lookup[image_id],
            'labels': label_vec,
            'modality': 'skin',
        })

    print(f'Loaded {len(records)} skin records')
    return records


def load_brain_data(config):
    """Load Brain Tumor Classification MRI dataset."""
    root = config['brain_root']
    if not os.path.isdir(root):
        print(f'Brain dataset not found at {root}, skipping.')
        return []

    folder_map = config['brain_folder_to_label']
    brain_labels = config['brain_labels']
    brain_offset = 21  # brain labels start at index 21

    records = []
    for split in ['Training', 'Testing']:
        split_dir = os.path.join(root, split)
        if not os.path.isdir(split_dir):
            continue
        for folder_name in os.listdir(split_dir):
            folder_path = os.path.join(split_dir, folder_name)
            if not os.path.isdir(folder_path):
                continue
            if folder_name not in folder_map:
                continue
            label_name = folder_map[folder_name]
            label_vec = np.zeros(config['num_classes'], dtype=np.float32)
            label_vec[brain_offset + brain_labels.index(label_name)] = 1.0

            for fname in os.listdir(folder_path):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    records.append({
                        'image_path': os.path.join(folder_path, fname),
                        'labels': label_vec,
                        'modality': 'brain',
                    })

    print(f'Loaded {len(records)} brain records')
    return records


# Load all datasets
all_records = []
all_records.extend(load_chest_data(CONFIG))
all_records.extend(load_skin_data(CONFIG))
all_records.extend(load_brain_data(CONFIG))

print(f'\nTotal records: {len(all_records)}')

# Count per modality
from collections import Counter
modality_counts = Counter(r['modality'] for r in all_records)
print(f'Per modality: {dict(modality_counts)}')

In [ ]:
# ── Train/Val Split ────────────────────────────────────────────────────────

modalities = [r['modality'] for r in all_records]
train_records, val_records = train_test_split(
    all_records,
    test_size=CONFIG['val_split'],
    random_state=SEED,
    stratify=modalities,
)

print(f'Train: {len(train_records)}, Val: {len(val_records)}')

In [ ]:
# ── Dataset Class ─────────────────────────────────────────────────────────

class MedicalImageDataset(Dataset):
    def __init__(self, records, transform=None, default_transform=None):
        self.records = records
        self.transform = transform
        self.default_transform = default_transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            image = Image.open(rec['image_path']).convert('RGB')
        except Exception:
            # Return a black image on failure
            image = Image.new('RGB', (CONFIG['image_size'], CONFIG['image_size']))

        if self.transform is not None:
            image = self.transform(image)
        elif self.default_transform is not None:
            image = self.default_transform(image)

        labels = torch.tensor(rec['labels'], dtype=torch.float32)

        # Modality index for modality classifier
        mod_map = {'chest': 0, 'skin': 1, 'brain': 2}
        modality_idx = mod_map[rec['modality']]

        return image, labels, modality_idx


# Build datasets
train_tf = get_train_transforms(CONFIG['image_size'])
val_tf = get_val_transforms(CONFIG['image_size'])

train_dataset = MedicalImageDataset(train_records, transform=train_tf)
val_dataset = MedicalImageDataset(val_records, transform=val_tf)

# ── WeightedRandomSampler for balanced modality representation ────────────
# Without this, chest images (~80% of data) dominate every batch and the model
# never gets enough skin/brain signal. We assign each sample a weight inversely
# proportional to its modality count so that each modality is equally likely.

modality_counts_train = Counter(r['modality'] for r in train_records)
mod_weights = {mod: 1.0 / count for mod, count in modality_counts_train.items()}
sample_weights = [mod_weights[r['modality']] for r in train_records]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_records),
    replacement=True,
)

# DataLoaders  (sampler is mutually exclusive with shuffle)
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    sampler=sampler,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
print(f'Modality counts (train): {dict(modality_counts_train)}')
print(f'Sampler weights: { {m: f"{w:.6f}" for m, w in mod_weights.items()} }')

## 4. Training Loop

In [ ]:
# ── Training Utilities ─────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []

    pbar = tqdm(loader, desc='Train', leave=False)
    for images, labels, _ in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.append(probs)
        all_targets.append(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, (all_preds >= 0.5).astype(int), average='macro', zero_division=0)

    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for images, labels, _ in tqdm(loader, desc='Val', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item()
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, (all_preds >= 0.5).astype(int), average='macro', zero_division=0)

    # ROC-AUC (only for classes with both pos and neg samples)
    valid_cols = [
        c for c in range(all_targets.shape[1])
        if all_targets[:, c].sum() > 0 and all_targets[:, c].sum() < len(all_targets)
    ]
    roc_auc = roc_auc_score(
        all_targets[:, valid_cols], all_preds[:, valid_cols], average='macro'
    ) if valid_cols else 0.0

    return avg_loss, f1, roc_auc, all_preds, all_targets

In [ ]:
# ── Train Image Encoder ───────────────────────────────────────────────────

# Build model
encoder = ImageEncoder(
    backbone_name=CONFIG['backbone'],
    pretrained=True,
    feature_dim=CONFIG['feature_dim'],
    projection_dim=CONFIG['projection_dim'],
    dropout=CONFIG['dropout'],
    freeze_backbone=True,  # Start frozen
)
img_classifier = ImageClassifier(encoder, CONFIG['num_classes'], CONFIG['dropout']).to(DEVICE)

criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])
scaler = GradScaler()

# ── Phase 1: Frozen backbone ──────────────────────────────────────────────
print('\n' + '='*60)
print('Phase 1: Training with FROZEN backbone')
print('='*60)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, img_classifier.parameters()),
    lr=CONFIG['lr_frozen'],
    weight_decay=CONFIG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs_frozen']
)

best_val_f1 = 0.0
history = []

for epoch in range(CONFIG['epochs_frozen']):
    train_loss, train_f1 = train_one_epoch(
        img_classifier, train_loader, criterion, optimizer, scaler, DEVICE
    )
    val_loss, val_f1, val_auc, _, _ = validate(
        img_classifier, val_loader, criterion, DEVICE
    )
    scheduler.step()

    history.append({
        'epoch': epoch + 1, 'phase': 'frozen',
        'train_loss': train_loss, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_f1': val_f1, 'val_auc': val_auc,
    })

    print(
        f'Epoch {epoch+1}/{CONFIG["epochs_frozen"]} (frozen) | '
        f'Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | '
        f'Val Loss: {val_loss:.4f} F1: {val_f1:.4f} AUC: {val_auc:.4f}'
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(encoder.state_dict(), os.path.join(CONFIG['output_dir'], 'image_encoder_best.pth'))


# ── Phase 2: Unfreeze last 2 blocks ───────────────────────────────────────
print('\n' + '='*60)
print('Phase 2: Training with UNFROZEN backbone (last 2 blocks)')
print('='*60)

encoder.unfreeze_backbone(last_n_blocks=2)

optimizer = torch.optim.AdamW(
    [
        {'params': encoder.projection.parameters(), 'lr': CONFIG['lr_unfrozen']},
        {'params': encoder.features.parameters(), 'lr': CONFIG['lr_unfrozen'] * 0.1},
        {'params': img_classifier.head.parameters(), 'lr': CONFIG['lr_unfrozen']},
    ],
    weight_decay=CONFIG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs_unfrozen']
)

for epoch in range(CONFIG['epochs_unfrozen']):
    train_loss, train_f1 = train_one_epoch(
        img_classifier, train_loader, criterion, optimizer, scaler, DEVICE
    )
    val_loss, val_f1, val_auc, _, _ = validate(
        img_classifier, val_loader, criterion, DEVICE
    )
    scheduler.step()

    history.append({
        'epoch': CONFIG['epochs_frozen'] + epoch + 1, 'phase': 'unfrozen',
        'train_loss': train_loss, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_f1': val_f1, 'val_auc': val_auc,
    })

    print(
        f'Epoch {epoch+1}/{CONFIG["epochs_unfrozen"]} (unfrozen) | '
        f'Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | '
        f'Val Loss: {val_loss:.4f} F1: {val_f1:.4f} AUC: {val_auc:.4f}'
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(encoder.state_dict(), os.path.join(CONFIG['output_dir'], 'image_encoder_best.pth'))

print(f'\nBest validation F1: {best_val_f1:.4f}')

## 5. Train Modality Classifier

In [ ]:
# ── Modality Classifier Training ──────────────────────────────────────────

# We only need the modality label, reuse same DataLoaders
mod_clf = ModalityClassifier(num_classes=3, pretrained=True).to(DEVICE)
mod_criterion = nn.CrossEntropyLoss()
mod_optimizer = torch.optim.AdamW(mod_clf.parameters(), lr=1e-3, weight_decay=0.01)
mod_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(mod_optimizer, T_max=10)

# We need 224x224 inputs for MobileNetV3. Let's create resized loaders.
mod_train_tf = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
mod_val_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

mod_train_ds = MedicalImageDataset(train_records, transform=mod_train_tf)
mod_val_ds = MedicalImageDataset(val_records, transform=mod_val_tf)

mod_train_loader = DataLoader(mod_train_ds, batch_size=32, shuffle=True,
                               num_workers=CONFIG['num_workers'], pin_memory=True)
mod_val_loader = DataLoader(mod_val_ds, batch_size=32, shuffle=False,
                             num_workers=CONFIG['num_workers'], pin_memory=True)

best_mod_acc = 0.0

for epoch in range(10):
    mod_clf.train()
    correct, total = 0, 0
    for images, _, mod_labels in tqdm(mod_train_loader, desc=f'ModClf Epoch {epoch+1}', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        mod_labels = mod_labels.to(DEVICE, non_blocking=True)

        mod_optimizer.zero_grad()
        logits = mod_clf(images)
        loss = mod_criterion(logits, mod_labels)
        loss.backward()
        mod_optimizer.step()

        preds = logits.argmax(dim=1)
        correct += (preds == mod_labels).sum().item()
        total += len(mod_labels)

    mod_scheduler.step()
    train_acc = correct / total

    # Validation
    mod_clf.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, _, mod_labels in mod_val_loader:
            images = images.to(DEVICE, non_blocking=True)
            mod_labels = mod_labels.to(DEVICE, non_blocking=True)
            logits = mod_clf(images)
            preds = logits.argmax(dim=1)
            correct += (preds == mod_labels).sum().item()
            total += len(mod_labels)

    val_acc = correct / total
    print(f'ModClf Epoch {epoch+1}/10 | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_mod_acc:
        best_mod_acc = val_acc
        torch.save(mod_clf.state_dict(), os.path.join(CONFIG['output_dir'], 'modality_classifier.pth'))

print(f'\nBest modality classifier accuracy: {best_mod_acc:.4f}')

## 6. Evaluation

In [ ]:
# ── Final Evaluation ──────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve

# Load best encoder weights
best_path = os.path.join(CONFIG['output_dir'], 'image_encoder_best.pth')
if os.path.exists(best_path):
    encoder.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print('Loaded best encoder weights.')

val_loss, val_f1, val_auc, all_preds, all_targets = validate(
    img_classifier, val_loader, criterion, DEVICE
)

# ── Per-class optimal threshold via precision-recall curve ────────────────
# Instead of a fixed 0.5 cutoff (which fails for rare chest classes where the
# model outputs low probabilities), we find the threshold that maximises F1
# for each class individually.

optimal_thresholds = np.full(CONFIG['num_classes'], 0.5)  # fallback

for i in range(CONFIG['num_classes']):
    pos_count = all_targets[:, i].sum()
    if pos_count < 2:
        continue  # too few positives to compute a meaningful curve
    precision, recall, thresholds = precision_recall_curve(
        all_targets[:, i], all_preds[:, i]
    )
    # F1 = 2 * P * R / (P + R)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_vals = np.where(
            (precision + recall) > 0,
            2 * precision * recall / (precision + recall),
            0.0,
        )
    best_idx = np.argmax(f1_vals[:-1])  # last entry is always (P=1, R=0)
    optimal_thresholds[i] = thresholds[best_idx]

print('Optimal per-class thresholds:')
for i, label in enumerate(CONFIG['all_labels']):
    if all_targets[:, i].sum() > 0:
        print(f'  {label:25s}  threshold={optimal_thresholds[i]:.3f}')

# ── Recompute metrics with optimal thresholds ────────────────────────────

y_bin_fixed = (all_preds >= 0.5).astype(int)
y_bin_opt = (all_preds >= optimal_thresholds[None, :]).astype(int)

f1_fixed = f1_score(all_targets, y_bin_fixed, average='macro', zero_division=0)
f1_opt = f1_score(all_targets, y_bin_opt, average='macro', zero_division=0)

print(f'\nFinal Validation Results:')
print(f'  Loss:                {val_loss:.4f}')
print(f'  ROC-AUC:             {val_auc:.4f}')
print(f'  Macro F1 (τ=0.5):   {f1_fixed:.4f}')
print(f'  Macro F1 (optimal τ): {f1_opt:.4f}')

# Per-class metrics with optimal thresholds
print(f'\nPer-class F1 scores (optimal thresholds):')
print('-' * 55)
for i, label in enumerate(CONFIG['all_labels']):
    if all_targets[:, i].sum() > 0:
        cls_f1_fixed = f1_score(all_targets[:, i], y_bin_fixed[:, i], zero_division=0)
        cls_f1_opt = f1_score(all_targets[:, i], y_bin_opt[:, i], zero_division=0)
        count = int(all_targets[:, i].sum())
        delta = cls_f1_opt - cls_f1_fixed
        marker = ' ▲' if delta > 0.01 else ''
        print(f'  {label:25s}  F1={cls_f1_opt:.4f}  (was {cls_f1_fixed:.4f}, τ={optimal_thresholds[i]:.3f}, n={count}){marker}')

# Per-modality summary
print(f'\nPer-modality ROC-AUC:')
print('-' * 40)
for mod, (start, end) in [('chest', (0, 14)), ('skin', (14, 21)), ('brain', (21, 25))]:
    cols = [c for c in range(start, end)
            if all_targets[:, c].sum() > 0 and all_targets[:, c].sum() < len(all_targets)]
    if cols:
        auc = roc_auc_score(all_targets[:, cols], all_preds[:, cols], average='macro')
        mod_f1 = f1_score(all_targets[:, cols], y_bin_opt[:, cols], average='macro', zero_division=0)
        print(f'  {mod:8s} AUC: {auc:.4f}  F1(opt): {mod_f1:.4f}')

In [ ]:
# ── Training History Plot ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
val_f1s = [h['val_f1'] for h in history]
val_aucs = [h['val_auc'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, train_losses, label='Train')
axes[0].plot(epochs, val_losses, label='Val')
axes[0].axvline(x=CONFIG['epochs_frozen'], color='red', linestyle='--', alpha=0.5, label='Unfreeze')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, val_f1s, 'g-')
axes[1].axvline(x=CONFIG['epochs_frozen'], color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1'); axes[1].set_title('Validation Macro F1')

axes[2].plot(epochs, val_aucs, 'm-')
axes[2].axvline(x=CONFIG['epochs_frozen'], color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC'); axes[2].set_title('Validation ROC-AUC')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_history.png'), dpi=150)
plt.show()

## 7. Save Models

In [ ]:
# ── Save Final Models ─────────────────────────────────────────────────────

output_dir = CONFIG['output_dir']

# Save encoder (final — may differ from best if last epoch was worse)
torch.save(encoder.state_dict(), os.path.join(output_dir, 'image_encoder.pth'))

# Save the image classification head separately (for single-modal fallback)
torch.save(img_classifier.head.state_dict(), os.path.join(output_dir, 'image_head.pth'))

# Config JSON (for loading locally)
image_config = {
    'backbone': CONFIG['backbone'],
    'feature_dim': CONFIG['feature_dim'],
    'projection_dim': CONFIG['projection_dim'],
    'num_classes': CONFIG['num_classes'],
    'dropout': CONFIG['dropout'],
    'image_size': CONFIG['image_size'],
    'all_labels': CONFIG['all_labels'],
    'optimal_thresholds': optimal_thresholds.tolist(),
    'best_val_f1': best_val_f1,
    'best_val_f1_optimal': float(f1_opt),
    'best_modality_acc': best_mod_acc,
    'training_history': history,
}

with open(os.path.join(output_dir, 'image_config.json'), 'w') as f:
    json.dump(image_config, f, indent=2)

# List output files
print('\nSaved files:')
for fname in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'  {fname:40s} {size_mb:.1f} MB')